In [ ]:
"""
================================================================================
CRESTA MLE INTERVIEW — PRACTICAL CODING PROBLEM
================================================================================
DOMAIN:  Contact-Center Intent Classification
TIME:    90 minutes
LEVEL:   Mid-Senior MLE

SCENARIO
--------
You are an MLE at Cresta. The Agent Assist product needs to classify customer
intent in real time during live calls so that the system can surface the right
knowledge articles and behavioral hints to human agents.

The contact center handles 5 intent categories:
  - billing         (billing disputes, payment issues, charges)
  - cancellation    (cancel service, close account, churn risk)
  - technical       (outage, device issue, connectivity)
  - account_change  (update address, change plan, add a line)
  - general_inquiry (store hours, coverage area, how-to questions)

CRITICAL BUSINESS CONTEXT
- "cancellation" is the highest-value class — even though it's rare (~5% of
  calls), missing one means losing a customer. The retention team needs 95%+
  recall on this class.
- Latency budget: inference must be < 50ms per utterance (rules out huge models
  at serving time without distillation or caching).
- New intent classes are added quarterly. Your system should be easy to extend.

YOUR TASK  (4 parts, increasing difficulty)
-------------------------------------------
Part 1: Data exploration & preprocessing            [15 min]
Part 2: Baseline classifier                         [25 min]
Part 3: Handle class imbalance for cancellation     [25 min]
Part 4: Evaluation + production considerations      [25 min]

Each part has a function stub. Fill them in. Run this file to verify.
================================================================================
"""

import random
import re
import json
from collections import Counter
from typing import Dict, List, Tuple, Optional

# ─────────────────────────────────────────────────────────────────────────────
# SYNTHETIC DATASET GENERATOR  (do NOT modify — this simulates the data you'd
# receive from Cresta's conversation pipeline)
# ─────────────────────────────────────────────────────────────────────────────

TEMPLATES = {
    "billing": [
        "I was charged twice for my last bill, can you help?",
        "Why is my bill higher than usual this month?",
        "I need to dispute a charge on my account",
        "Can you explain the fees on my latest statement?",
        "I made a payment but it's not showing up",
        "There's an unauthorized charge on my account",
        "I want to set up autopay for my monthly bill",
        "My promotional rate expired and my bill jumped",
        "Can I get an extension on my payment due date?",
        "I was promised a credit but it hasn't been applied",
        "The late fee on my account isn't fair, I paid on time",
        "I need a copy of my billing statement from last month",
        "Why am I being charged for a service I never ordered?",
        "Can you break down my charges line by line?",
        "I want to switch to paperless billing",
    ],
    "cancellation": [
        "I want to cancel my service effective immediately",
        "I'm switching to a competitor, please close my account",
        "Cancel everything on my account",
        "I've had enough, I want to terminate my contract",
        "How do I go about cancelling my subscription?",
        "I no longer need this service, please discontinue it",
        "I'd like to close my account permanently",
        "Your service isn't worth what I'm paying, I want out",
        "Please process my cancellation request",
        "I'm done with this company, cancel it all",
        "I want to end my service at the end of this billing cycle",
        "What are the fees for early termination?",
    ],
    "technical": [
        "My internet has been down since this morning",
        "I'm getting really slow speeds on my connection",
        "The app keeps crashing when I try to log in",
        "My device won't connect to the network",
        "Is there a service outage in my area?",
        "I keep getting an error message when I try to stream",
        "My router needs to be reset but I don't know how",
        "The signal keeps dropping in and out",
        "I can't send or receive messages on my device",
        "My equipment is making a strange buzzing noise",
        "I just installed the new update and nothing works now",
        "Can you run a diagnostic on my connection?",
        "My TV box is frozen and won't respond to the remote",
        "I need help setting up my new modem",
    ],
    "account_change": [
        "I need to update my mailing address",
        "I want to upgrade my plan to the premium tier",
        "Can I add another user to my account?",
        "I'm moving to a new city, can I transfer my service?",
        "I need to change the name on my account",
        "I want to downgrade my plan to save money",
        "Can you update my email address on file?",
        "I'd like to add international calling to my plan",
        "I need to remove a line from my family plan",
        "Can I change my billing date to the 15th?",
        "I want to add the sports package to my subscription",
        "I need to update my payment method to a new card",
    ],
    "general_inquiry": [
        "What are your store hours this weekend?",
        "Do you offer service in the Portland area?",
        "How do I check my current data usage?",
        "What's the difference between your basic and premium plans?",
        "Where can I find my account number?",
        "Do you have any current promotions going on?",
        "How long is the warranty on this equipment?",
        "Can I use my service when traveling internationally?",
        "What's your return policy for equipment?",
        "How do I access my account online?",
        "Is there a loyalty program for long-term customers?",
        "What channels are included in the basic package?",
        "How do I contact technical support after hours?",
        "Do you offer any military or senior discounts?",
    ],
}

# Noise augmentations to make the data more realistic
PREFIXES = [
    "", "", "", "",  # most have no prefix
    "Yeah so ", "Um ", "Hi, ", "Hey, ", "Look, ", "So basically ",
    "I'm calling because ", "Hi there, ", "Hello, ",
]
SUFFIXES = [
    "", "", "", "",
    " Thanks.", " Please help.", " This is urgent.", " I've been waiting.",
    " Can someone help me?", " I need this resolved today.",
]
TYPO_PATTERNS = [
    (r"I'm", "Im"), (r"I've", "Ive"), (r"can't", "cant"),
    (r"don't", "dont"), (r"isn't", "isnt"), (r"won't", "wont"),
]


def _augment(text: str, rng: random.Random) -> str:
    """Add realistic noise to a template utterance."""
    text = rng.choice(PREFIXES) + text + rng.choice(SUFFIXES)
    if rng.random() < 0.15:
        text = text.lower()
    if rng.random() < 0.10:
        for pat, rep in TYPO_PATTERNS:
            if rng.random() < 0.3:
                text = re.sub(pat, rep, text)
    return text.strip()


def generate_dataset(
    n_samples: int = 1000,
    seed: int = 42,
    imbalance: bool = True,
) -> Tuple[List[str], List[str]]:
    """
    Generate a synthetic contact-center intent dataset.

    Returns (texts, labels) where:
      - texts: list of customer utterances
      - labels: list of intent labels

    If imbalance=True (default), class distribution mimics production:
      billing: 30%, technical: 28%, general_inquiry: 22%,
      account_change: 15%, cancellation: 5%
    """
    rng = random.Random(seed)

    if imbalance:
        weights = {
            "billing": 0.30,
            "technical": 0.28,
            "general_inquiry": 0.22,
            "account_change": 0.15,
            "cancellation": 0.05,
        }
    else:
        weights = {k: 1 / len(TEMPLATES) for k in TEMPLATES}

    texts, labels = [], []
    for _ in range(n_samples):
        label = rng.choices(
            list(weights.keys()), weights=list(weights.values()), k=1
        )[0]
        template = rng.choice(TEMPLATES[label])
        text = _augment(template, rng)
        texts.append(text)
        labels.append(label)

    return texts, labels


# ─────────────────────────────────────────────────────────────────────────────
# PART 1: DATA EXPLORATION & PREPROCESSING                        [15 minutes]
# ─────────────────────────────────────────────────────────────────────────────
#
# Goals:
#   a) Load the dataset and print class distribution.
#   b) Implement a simple text preprocessing function.
#   c) Split into train/val/test with stratification.
#
# Why this matters at Cresta:
#   Before building any model, you need to understand the data distribution.
#   The cancellation class is deliberately underrepresented — you must notice
#   this and flag it as a risk.
# ─────────────────────────────────────────────────────────────────────────────

def explore_dataset(texts: List[str], labels: List[str]) -> Dict:
    """
    Return a dict summarizing the dataset:
    {
        "total_samples": int,
        "class_counts": {label: count, ...},
        "class_pcts": {label: float (0-100), ...},
        "avg_text_length": float,  # avg character length
        "shortest_text": str,
        "longest_text": str,
    }
    """
    # TODO: Implement this
    res = {}
    res['total_samples'] = len(texts)
    class_counts = {}
    for i, v in zip(texts, labels):
        class_counts[v]=class_counts.get(v,0)+1
    res['class_counts'] = class_counts

    avg_text_length1 = np.array([len(t) for t in texts])
    avg_text_length = avg_text_length1.mean()
    res['avg_text_length'] = avg_text_length

    shortest_text = sorted(texts, key = lambda x:len(x) )[0]
    longest_text = sorted(texts, key = lambda x:-len(x) )[0]
    res['shortest_text'] = shortest_text
    res['longest_text'] = longest_text

    raise NotImplementedError("Part 1a: Implement explore_dataset")


def preprocess_text(text: str) -> str:
    """
    Clean a single utterance for downstream classification.

    Minimum expectations:
      - Lowercase
      - Strip extra whitespace
      - Normalize common contractions (e.g., "cant" -> "can not")

    Bonus:
      - Remove filler words ("um", "yeah", "so basically")
      - Normalize punctuation
    """
    # TODO: Implement this
    raise NotImplementedError("Part 1b: Implement preprocess_text")


def stratified_split(
    texts: List[str],
    labels: List[str],
    train_ratio: float = 0.7,
    val_ratio: float = 0.15,
    test_ratio: float = 0.15,
    seed: int = 42,
) -> Dict[str, Tuple[List[str], List[str]]]:
    """
    Split data into train/val/test sets with stratified sampling.

    Returns:
    {
        "train": (texts, labels),
        "val": (texts, labels),
        "test": (texts, labels),
    }

    IMPORTANT: The cancellation class has only ~50 samples in 1000.
    Your split must preserve the class ratio in every partition so that
    the val/test sets still contain cancellation examples.
    """
    # TODO: Implement this
    raise NotImplementedError("Part 1c: Implement stratified_split")


# ─────────────────────────────────────────────────────────────────────────────
# PART 2: BASELINE CLASSIFIER                                     [25 minutes]
# ─────────────────────────────────────────────────────────────────────────────
#
# Build a TF-IDF + Logistic Regression baseline. This is your "can it even
# be solved?" check and your latency-friendly production fallback.
#
# Why this matters at Cresta:
#   Agent Assist has a <50ms latency budget. A TF-IDF + LR model serves in
#   <5ms. Even if you later use an LLM, you need this baseline to benchmark
#   against and as a fallback if the LLM is too slow or too expensive.
# ─────────────────────────────────────────────────────────────────────────────

def build_tfidf_baseline(
    train_texts: List[str],
    train_labels: List[str],
    val_texts: List[str],
    val_labels: List[str],
) -> Dict:
    """
    Train a TF-IDF + Logistic Regression classifier.

    Returns a dict:
    {
        "model": trained sklearn Pipeline or similar,
        "val_accuracy": float,
        "val_macro_f1": float,
        "val_per_class_f1": {label: float, ...},
        "val_predictions": List[str],    # predictions on val set
    }

    Requirements:
      - Use sklearn's TfidfVectorizer + LogisticRegression
      - Tune at least: ngram_range, max_features, C (regularization)
      - Report per-class F1 (you MUST flag if cancellation F1 is low)
      - Hint: solver='lbfgs' works well for multinomial classification
    """
    # TODO: Implement this
    raise NotImplementedError("Part 2: Implement build_tfidf_baseline")


# ─────────────────────────────────────────────────────────────────────────────
# PART 3: HANDLE CLASS IMBALANCE                                   [25 minutes]
# ─────────────────────────────────────────────────────────────────────────────
#
# The baseline likely has poor recall on "cancellation". Fix it.
# Implement AT LEAST TWO of the following strategies:
#
#   Strategy A: Class-weighted loss
#   Strategy B: Oversampling the minority class (SMOTE or random)
#   Strategy C: Threshold tuning on the cancellation class
#   Strategy D: A custom approach (e.g., two-stage classifier, cost-sensitive)
#
# Why this matters at Cresta:
#   Missing a cancellation call means the retention team never gets a chance to
#   save the customer. The business explicitly requires >=95% recall on
#   cancellation, even if it costs some precision on other classes.
# ─────────────────────────────────────────────────────────────────────────────

def improve_cancellation_recall(
    train_texts: List[str],
    train_labels: List[str],
    val_texts: List[str],
    val_labels: List[str],
    target_recall: float = 0.95,
) -> Dict:
    """
    Improve cancellation recall to >= target_recall on the val set.

    Returns a dict:
    {
        "strategies_used": List[str],    # e.g. ["class_weight", "threshold"]
        "model": trained model/pipeline,
        "cancellation_recall": float,    # must be >= 0.95
        "cancellation_precision": float,
        "val_macro_f1": float,
        "val_per_class_report": str,     # sklearn classification_report
        "tradeoff_analysis": str,        # 2-3 sentences: what did you trade?
    }

    HINT: You can get predict_proba from LogisticRegression, then adjust the
    decision threshold for the cancellation class specifically.
    """
    # TODO: Implement this
    raise NotImplementedError("Part 3: Implement improve_cancellation_recall")


# ─────────────────────────────────────────────────────────────────────────────
# PART 4: EVALUATION + PRODUCTION READINESS                        [25 minutes]
# ─────────────────────────────────────────────────────────────────────────────
#
# Now evaluate your best model properly and discuss production considerations.
#
# Why this matters at Cresta:
#   Cresta's QM product auto-scores 100% of conversations. They care deeply
#   about evaluation rigor. A model that "works on the test set" but fails
#   silently in production is worse than no model.
# ─────────────────────────────────────────────────────────────────────────────

def full_evaluation(
    model,
    test_texts: List[str],
    test_labels: List[str],
    label_names: List[str],
) -> Dict:
    """
    Run comprehensive evaluation on the held-out test set.

    Returns a dict:
    {
        "test_accuracy": float,
        "test_macro_f1": float,
        "test_weighted_f1": float,
        "classification_report": str,
        "confusion_matrix": List[List[int]],
        "per_class_metrics": {
            label: {"precision": float, "recall": float, "f1": float}
        },
        "error_analysis": List[Dict],  # 5 most interesting misclassifications
            # each: {"text": str, "true": str, "predicted": str, "why": str}
    }
    """
    # TODO: Implement this
    raise NotImplementedError("Part 4a: Implement full_evaluation")


def production_design() -> Dict:
    """
    No code needed — return a dict answering these production design questions.
    Write your answers as strings.

    This is what separates a junior from a senior MLE at Cresta.
    """
    return {
        "latency_strategy": "",
        # How would you serve this model at < 50ms per utterance?
        # What's your inference architecture?

        "new_intent_handling": "",
        # A new intent class ("fraud_report") appears next quarter with only
        # 20 labeled examples. How do you add it without retraining from
        # scratch? Compare: few-shot prompting, fine-tuning, embedding
        # similarity, and updating your TF-IDF model.

        "monitoring_plan": "",
        # What metrics do you monitor in production? How do you detect
        # model drift (e.g., customers start using new vocabulary)?
        # What triggers a retrain?

        "escalation_policy": "",
        # When should your classifier say "I don't know" instead of
        # guessing? How do you implement a confidence threshold, and
        # what happens when it fires? (Think: Cresta's Agent Assist
        # would escalate to a human or ask a clarifying question.)

        "llm_vs_classical": "",
        # Your TF-IDF baseline runs in <5ms. An LLM-based classifier
        # (GPT-4 / Claude) gives +3% macro-F1 but costs 200ms + $0.002
        # per call. At 50,000 calls/day, what's your recommendation?
        # Is there a hybrid approach?
    }


# ─────────────────────────────────────────────────────────────────────────────
# RUNNER — execute this file to verify your implementations
# ─────────────────────────────────────────────────────────────────────────────

def main():
    print("=" * 70)
    print("CRESTA MLE INTERVIEW — Intent Classification")
    print("=" * 70)

    # Generate data
    texts, labels = generate_dataset(n_samples=1000, seed=42)
    label_names = sorted(set(labels))

    # ── Part 1 ──────────────────────────────────────────────────────────
    print("\n📊 PART 1: Data Exploration")
    print("-" * 40)
    try:
        summary = explore_dataset(texts, labels)
        print(f"  Total samples: {summary['total_samples']}")
        print(f"  Class distribution:")
        for cls, pct in sorted(
            summary["class_pcts"].items(), key=lambda x: -x[1]
        ):
            bar = "█" * int(pct / 2)
            print(f"    {cls:<20s} {summary['class_counts'][cls]:>4d} ({pct:5.1f}%) {bar}")
        print(f"  Avg text length: {summary['avg_text_length']:.1f} chars")
        print("  ✅ Part 1a passed")
    except NotImplementedError:
        print("  ⬜ Part 1a: Not implemented yet")

    try:
        cleaned = preprocess_text("Um I cant believe Im being charged TWICE!!!")
        assert isinstance(cleaned, str)
        assert cleaned == cleaned.strip()
        print(f'  Preprocessed: "{cleaned}"')
        print("  ✅ Part 1b passed")
    except NotImplementedError:
        print("  ⬜ Part 1b: Not implemented yet")

    try:
        splits = stratified_split(texts, labels)
        for name, (t, l) in splits.items():
            dist = Counter(l)
            canc_pct = dist.get("cancellation", 0) / len(l) * 100
            print(f"  {name:>5s}: {len(t):>4d} samples, cancellation={canc_pct:.1f}%")
        print("  ✅ Part 1c passed")
    except NotImplementedError:
        print("  ⬜ Part 1c: Not implemented yet")

    # ── Part 2 ──────────────────────────────────────────────────────────
    print("\n🤖 PART 2: Baseline Classifier")
    print("-" * 40)
    try:
        splits = stratified_split(texts, labels)
        train_t, train_l = splits["train"]
        val_t, val_l = splits["val"]
        test_t, test_l = splits["test"]

        # Preprocess
        train_t = [preprocess_text(t) for t in train_t]
        val_t = [preprocess_text(t) for t in val_t]
        test_t = [preprocess_text(t) for t in test_t]

        result = build_tfidf_baseline(train_t, train_l, val_t, val_l)
        print(f"  Val accuracy:  {result['val_accuracy']:.3f}")
        print(f"  Val macro F1:  {result['val_macro_f1']:.3f}")
        print(f"  Per-class F1:")
        for cls, f1 in sorted(result["val_per_class_f1"].items()):
            flag = " ⚠️  LOW" if f1 < 0.7 and cls == "cancellation" else ""
            print(f"    {cls:<20s} {f1:.3f}{flag}")
        print("  ✅ Part 2 passed")
    except NotImplementedError:
        print("  ⬜ Part 2: Not implemented yet")
    except Exception as e:
        print(f"  ❌ Part 2 error: {e}")

    # ── Part 3 ──────────────────────────────────────────────────────────
    print("\n⚖️  PART 3: Imbalance Handling")
    print("-" * 40)
    result3 = None
    try:
        if "train_t" not in dir():
            splits = stratified_split(texts, labels)
            train_t, train_l = splits["train"]
            val_t, val_l = splits["val"]
            test_t, test_l = splits["test"]
            train_t = [preprocess_text(t) for t in train_t]
            val_t = [preprocess_text(t) for t in val_t]
            test_t = [preprocess_text(t) for t in test_t]
        result3 = improve_cancellation_recall(
            train_t, train_l, val_t, val_l, target_recall=0.95
        )
        canc_r = result3["cancellation_recall"]
        canc_p = result3["cancellation_precision"]
        status = "✅ TARGET MET" if canc_r >= 0.95 else "❌ BELOW TARGET"
        print(f"  Strategies: {', '.join(result3['strategies_used'])}")
        print(f"  Cancellation recall:    {canc_r:.3f}  {status}")
        print(f"  Cancellation precision: {canc_p:.3f}")
        print(f"  Macro F1:               {result3['val_macro_f1']:.3f}")
        print(f"  Tradeoff: {result3['tradeoff_analysis']}")
        print("  ✅ Part 3 passed" if canc_r >= 0.95 else "  ⚠️  Part 3: target not met")
    except NotImplementedError:
        print("  ⬜ Part 3: Not implemented yet")
    except Exception as e:
        print(f"  ❌ Part 3 error: {e}")

    # ── Part 4 ──────────────────────────────────────────────────────────
    print("\n📋 PART 4: Evaluation + Production Design")
    print("-" * 40)
    try:
        if result3 is not None and result3.get("model"):
            eval_result = full_evaluation(
                result3["model"], test_t, test_l, label_names
            )
            print(f"  Test accuracy:    {eval_result['test_accuracy']:.3f}")
            print(f"  Test macro F1:    {eval_result['test_macro_f1']:.3f}")
            print(f"  Test weighted F1: {eval_result['test_weighted_f1']:.3f}")
            print(f"\n  Error analysis (sample):")
            for err in eval_result["error_analysis"][:3]:
                print(f'    "{err["text"][:60]}..."')
                print(f'      True: {err["true"]}  →  Predicted: {err["predicted"]}')
                print(f'      Why: {err["why"]}')
            print("  ✅ Part 4a passed")
    except NotImplementedError:
        print("  ⬜ Part 4a: Not implemented yet")
    except Exception as e:
        print(f"  ❌ Part 4a error: {e}")

    try:
        design = production_design()
        filled = sum(1 for v in design.values() if v.strip())
        total = len(design)
        print(f"\n  Production design: {filled}/{total} questions answered")
        if filled == total:
            print("  ✅ Part 4b passed")
        else:
            missing = [k for k, v in design.items() if not v.strip()]
            print(f"  ⬜ Missing: {', '.join(missing)}")
    except Exception as e:
        print(f"  ❌ Part 4b error: {e}")

    print("\n" + "=" * 70)
    print("DONE — Review output above for pass/fail status on each part.")
    print("=" * 70)


if __name__ == "__main__":
    main()

In [ ]:
"""
================================================================================
REFERENCE SOLUTION — Cresta MLE Intent Classification
================================================================================
IMPORTANT: Attempt the problem yourself first! Only check this after you've
spent the full 90 minutes or gotten stuck on a specific part.
================================================================================
"""

import random
import re
from collections import Counter, defaultdict
from typing import Dict, List, Tuple, Optional

# Import the problem file for dataset generation and templates
from cresta_intent_classification_problem import (
    generate_dataset,
    TEMPLATES,
)

# ─── sklearn imports ────────────────────────────────────────────────────────
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
import numpy as np


# ═════════════════════════════════════════════════════════════════════════════
# PART 1: Data Exploration & Preprocessing
# ═════════════════════════════════════════════════════════════════════════════

def explore_dataset(texts: List[str], labels: List[str]) -> Dict:
    counts = Counter(labels)
    total = len(texts)
    lengths = [len(t) for t in texts]

    return {
        "total_samples": total,
        "class_counts": dict(counts),
        "class_pcts": {k: (v / total) * 100 for k, v in counts.items()},
        "avg_text_length": sum(lengths) / len(lengths),
        "shortest_text": min(texts, key=len),
        "longest_text": max(texts, key=len),
    }


CONTRACTION_MAP = {
    "cant": "can not",
    "dont": "don not",
    "wont": "will not",
    "isnt": "is not",
    "im": "i am",
    "ive": "i have",
    "didnt": "did not",
    "doesnt": "does not",
    "havent": "have not",
    "wouldnt": "would not",
    "shouldnt": "should not",
}

FILLER_WORDS = {"um", "uh", "yeah", "so", "basically", "like", "well"}


def preprocess_text(text: str) -> str:
    # Lowercase
    text = text.lower().strip()

    # Remove "so basically" as a unit first
    text = re.sub(r"\bso basically\b", "", text)

    # Normalize contractions
    words = text.split()
    words = [CONTRACTION_MAP.get(w, w) for w in words]

    # Remove filler words at the start
    while words and words[0] in FILLER_WORDS:
        words.pop(0)

    # Rejoin and clean up whitespace/punctuation
    text = " ".join(words)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def stratified_split(
    texts: List[str],
    labels: List[str],
    train_ratio: float = 0.7,
    val_ratio: float = 0.15,
    test_ratio: float = 0.15,
    seed: int = 42,
) -> Dict[str, Tuple[List[str], List[str]]]:
    """Stratified split preserving class proportions in each partition."""
    rng = random.Random(seed)

    # Group indices by label
    label_to_indices = defaultdict(list)
    for i, label in enumerate(labels):
        label_to_indices[label].append(i)

    train_idx, val_idx, test_idx = [], [], []

    for label, indices in label_to_indices.items():
        rng.shuffle(indices)
        n = len(indices)
        n_train = max(1, int(n * train_ratio))
        n_val = max(1, int(n * val_ratio))
        # Rest goes to test

        train_idx.extend(indices[:n_train])
        val_idx.extend(indices[n_train : n_train + n_val])
        test_idx.extend(indices[n_train + n_val :])

    # Shuffle within each split
    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    rng.shuffle(test_idx)

    return {
        "train": ([texts[i] for i in train_idx], [labels[i] for i in train_idx]),
        "val": ([texts[i] for i in val_idx], [labels[i] for i in val_idx]),
        "test": ([texts[i] for i in test_idx], [labels[i] for i in test_idx]),
    }


# ═════════════════════════════════════════════════════════════════════════════
# PART 2: Baseline Classifier
# ═════════════════════════════════════════════════════════════════════════════

def build_tfidf_baseline(
    train_texts: List[str],
    train_labels: List[str],
    val_texts: List[str],
    val_labels: List[str],
) -> Dict:
    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(
            ngram_range=(1, 2),     # Unigrams + bigrams capture phrases like "cancel service"
            max_features=5000,      # Enough for this vocabulary size
            min_df=2,               # Remove very rare terms
            sublinear_tf=True,      # Dampen term frequency (log scaling)
        )),
        ("clf", LogisticRegression(
            C=1.0,                  # Default regularization
            max_iter=1000,
            solver="lbfgs",
            random_state=42,
        )),
    ])

    pipeline.fit(train_texts, train_labels)
    val_preds = pipeline.predict(val_texts)

    # Per-class F1
    labels_unique = sorted(set(train_labels))
    precisions, recalls, f1s, _ = precision_recall_fscore_support(
        val_labels, val_preds, labels=labels_unique, zero_division=0
    )
    per_class_f1 = dict(zip(labels_unique, f1s))

    return {
        "model": pipeline,
        "val_accuracy": accuracy_score(val_labels, val_preds),
        "val_macro_f1": f1_score(val_labels, val_preds, average="macro"),
        "val_per_class_f1": per_class_f1,
        "val_predictions": list(val_preds),
    }


# ═════════════════════════════════════════════════════════════════════════════
# PART 3: Handle Class Imbalance
# ═════════════════════════════════════════════════════════════════════════════

def improve_cancellation_recall(
    train_texts: List[str],
    train_labels: List[str],
    val_texts: List[str],
    val_labels: List[str],
    target_recall: float = 0.95,
) -> Dict:
    """
    Two strategies combined:
      A) class_weight='balanced' in LogisticRegression
      C) Threshold tuning on the cancellation class probability
    """
    label_names = sorted(set(train_labels))

    # Strategy A: Class-weighted loss
    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(
            ngram_range=(1, 2),
            max_features=5000,
            min_df=2,
            sublinear_tf=True,
        )),
        ("clf", LogisticRegression(
            C=1.0,
            class_weight="balanced",   # <-- KEY CHANGE
            max_iter=1000,
            solver="lbfgs",
            random_state=42,
        )),
    ])
    pipeline.fit(train_texts, train_labels)

    # Strategy C: Threshold tuning
    # Get probabilities and find the cancellation class index
    probas = pipeline.predict_proba(val_texts)
    canc_idx = list(pipeline.classes_).index("cancellation")

    # Default predictions
    default_preds = pipeline.predict(val_texts)

    # Tune threshold: lower the threshold for cancellation so we catch more
    # Try thresholds from 0.5 down to 0.05
    best_threshold = 0.5
    best_preds = default_preds

    for threshold in [t / 100 for t in range(50, 4, -1)]:
        preds = list(default_preds)  # start from default
        for i in range(len(val_texts)):
            if probas[i][canc_idx] >= threshold:
                preds[i] = "cancellation"

        # Check cancellation recall
        canc_true = [1 if l == "cancellation" else 0 for l in val_labels]
        canc_pred = [1 if p == "cancellation" else 0 for p in preds]
        canc_recall = sum(t == 1 and p == 1 for t, p in zip(canc_true, canc_pred)) / max(
            sum(canc_true), 1
        )
        if canc_recall >= target_recall:
            best_threshold = threshold
            best_preds = preds
            break

    # Compute final metrics
    final_preds = best_preds
    canc_true = [1 if l == "cancellation" else 0 for l in val_labels]
    canc_pred = [1 if p == "cancellation" else 0 for p in final_preds]

    canc_recall = sum(t == 1 and p == 1 for t, p in zip(canc_true, canc_pred)) / max(
        sum(canc_true), 1
    )
    canc_precision = sum(t == 1 and p == 1 for t, p in zip(canc_true, canc_pred)) / max(
        sum(canc_pred), 1
    )

    report = classification_report(val_labels, final_preds, labels=label_names)

    tradeoff = (
        f"Used class_weight='balanced' + threshold tuning (threshold={best_threshold:.2f}). "
        f"Cancellation recall reached {canc_recall:.2f}, but precision dropped to {canc_precision:.2f} — "
        f"some billing/account_change utterances now get misclassified as cancellation. "
        f"This is acceptable: a false cancellation alert wastes a few seconds of agent attention, "
        f"but missing a real cancellation loses a customer."
    )

    # Wrap the model + threshold into a simple callable object for Part 4
    class ThresholdModel:
        def __init__(self, pipeline, threshold, canc_idx):
            self.pipeline = pipeline
            self.threshold = threshold
            self.canc_idx = canc_idx
            self.classes_ = pipeline.classes_

        def predict(self, texts):
            probas = self.pipeline.predict_proba(texts)
            preds = list(self.pipeline.predict(texts))
            for i in range(len(texts)):
                if probas[i][self.canc_idx] >= self.threshold:
                    preds[i] = "cancellation"
            return preds

        def predict_proba(self, texts):
            return self.pipeline.predict_proba(texts)

    threshold_model = ThresholdModel(pipeline, best_threshold, canc_idx)

    return {
        "strategies_used": ["class_weight_balanced", "threshold_tuning"],
        "model": threshold_model,
        "cancellation_recall": canc_recall,
        "cancellation_precision": canc_precision,
        "val_macro_f1": f1_score(val_labels, final_preds, average="macro"),
        "val_per_class_report": report,
        "tradeoff_analysis": tradeoff,
    }


# ═════════════════════════════════════════════════════════════════════════════
# PART 4: Evaluation + Production Design
# ═════════════════════════════════════════════════════════════════════════════

def full_evaluation(
    model,
    test_texts: List[str],
    test_labels: List[str],
    label_names: List[str],
) -> Dict:
    preds = model.predict(test_texts)

    # Per-class metrics
    precisions, recalls, f1s, _ = precision_recall_fscore_support(
        test_labels, preds, labels=label_names, zero_division=0
    )
    per_class = {}
    for i, name in enumerate(label_names):
        per_class[name] = {
            "precision": round(float(precisions[i]), 3),
            "recall": round(float(recalls[i]), 3),
            "f1": round(float(f1s[i]), 3),
        }

    # Error analysis: find misclassifications
    errors = []
    for text, true, pred in zip(test_texts, test_labels, preds):
        if true != pred:
            # Generate a brief explanation of why this might be confusing
            if true == "cancellation" and pred == "billing":
                why = "Mentions payment/charges — overlaps with billing vocabulary"
            elif true == "billing" and pred == "cancellation":
                why = "Threshold tuning is aggressive — any payment frustration triggers cancellation"
            elif true == "account_change" and pred == "billing":
                why = "Account changes involving plan/payment overlap with billing"
            elif true == "general_inquiry" and pred == "technical":
                why = "Question about service capabilities sounds like a tech issue"
            else:
                why = f"Vocabulary overlap between {true} and {pred}"
            errors.append({
                "text": text,
                "true": true,
                "predicted": pred,
                "why": why,
            })

    # Sort errors to surface the most interesting ones first (cancellation misses)
    errors.sort(key=lambda e: (e["true"] != "cancellation", e["true"]))

    cm = confusion_matrix(test_labels, preds, labels=label_names)

    return {
        "test_accuracy": round(accuracy_score(test_labels, preds), 3),
        "test_macro_f1": round(f1_score(test_labels, preds, average="macro"), 3),
        "test_weighted_f1": round(
            f1_score(test_labels, preds, average="weighted"), 3
        ),
        "classification_report": classification_report(
            test_labels, preds, labels=label_names
        ),
        "confusion_matrix": cm.tolist(),
        "per_class_metrics": per_class,
        "error_analysis": errors[:5],
    }


def production_design() -> Dict:
    return {
        "latency_strategy": (
            "Serve the TF-IDF + LR model as an ONNX-exported artifact behind a "
            "FastAPI endpoint. TF-IDF vectorization + LR prediction runs in <5ms "
            "on CPU. No GPU needed. Load the vectorizer and model at startup into "
            "memory. For multi-model ensemble, run in parallel with asyncio. "
            "Batch multiple utterances from the same call if buffered."
        ),
        "new_intent_handling": (
            "For 20 examples of 'fraud_report': (1) Immediately deploy a "
            "few-shot LLM classifier as a temporary gate — high cost but fast "
            "to set up. (2) Use those 20 examples + data augmentation (paraphrase "
            "with an LLM) to expand to ~200 examples. (3) Retrain the TF-IDF + LR "
            "model incrementally by warm-starting from existing weights. (4) Use "
            "embedding similarity (sentence-transformers) as a complementary signal "
            "— if cosine similarity to the 20 examples exceeds a threshold, "
            "classify as fraud. The hybrid approach (embedding fallback + LR) "
            "balances speed and accuracy while the labeled set grows."
        ),
        "monitoring_plan": (
            "Track in production: (a) prediction distribution drift — if "
            "cancellation predictions drop below 4% or spike above 8%, alert. "
            "(b) Confidence calibration — log mean confidence per class weekly. "
            "(c) Downstream outcome correlation — do 'cancellation' predictions "
            "actually lead to retention offers? If not, the model may be stale. "
            "(d) Token/vocabulary drift — track OOV rate in TF-IDF; if >15% of "
            "tokens are unseen, trigger retraining. Retrain monthly on a rolling "
            "window of labeled data, validated against a golden test set."
        ),
        "escalation_policy": (
            "Use predict_proba to compute the margin between the top-1 and top-2 "
            "predictions. If margin < 0.15, flag as 'low confidence' and either: "
            "(a) surface to the human agent as a soft suggestion rather than a "
            "definitive label, or (b) trigger a secondary LLM classifier for "
            "just those ambiguous cases (paying the latency cost only when needed). "
            "For the cancellation class specifically, even a low-confidence "
            "cancellation signal should trigger a retention workflow — the cost of "
            "missing is too high."
        ),
        "llm_vs_classical": (
            "At 50,000 calls/day: LLM cost = 50K * $0.002 = $100/day = $36,500/yr. "
            "Plus 200ms adds latency that may degrade real-time agent assist. "
            "Recommendation: use TF-IDF + LR as the primary real-time classifier "
            "(5ms, $0). Use the LLM as a (a) batch re-scorer on low-confidence "
            "predictions (maybe 10% of calls → $3,650/yr), and (b) labeling tool "
            "to generate training data for retraining the fast model. The +3% F1 "
            "likely concentrates in edge cases that the threshold-tuned model "
            "already handles via the confidence escalation policy. Hybrid wins."
        ),
    }


# ─────────────────────────────────────────────────────────────────────────────
# RUNNER
# ─────────────────────────────────────────────────────────────────────────────

def main():
    print("=" * 70)
    print("REFERENCE SOLUTION — Cresta MLE Intent Classification")
    print("=" * 70)

    texts, labels = generate_dataset(n_samples=1000, seed=42)
    label_names = sorted(set(labels))

    # Part 1
    print("\n📊 PART 1")
    summary = explore_dataset(texts, labels)
    for cls, pct in sorted(summary["class_pcts"].items(), key=lambda x: -x[1]):
        bar = "█" * int(pct / 2)
        print(f"  {cls:<20s} {summary['class_counts'][cls]:>4d} ({pct:5.1f}%) {bar}")

    cleaned = preprocess_text("Um I cant believe Im being charged TWICE!!!")
    print(f'\n  Preprocessed: "{cleaned}"')

    splits = stratified_split(texts, labels)
    train_t, train_l = splits["train"]
    val_t, val_l = splits["val"]
    test_t, test_l = splits["test"]

    train_t = [preprocess_text(t) for t in train_t]
    val_t = [preprocess_text(t) for t in val_t]
    test_t = [preprocess_text(t) for t in test_t]

    for name, (t, l) in splits.items():
        dist = Counter(l)
        canc_pct = dist.get("cancellation", 0) / len(l) * 100
        print(f"  {name:>5s}: {len(t)} samples, cancellation={canc_pct:.1f}%")

    # Part 2
    print("\n🤖 PART 2")
    baseline = build_tfidf_baseline(train_t, train_l, val_t, val_l)
    print(f"  Accuracy: {baseline['val_accuracy']:.3f}")
    print(f"  Macro F1: {baseline['val_macro_f1']:.3f}")
    for cls, f1 in sorted(baseline["val_per_class_f1"].items()):
        flag = " ⚠️" if f1 < 0.7 else ""
        print(f"    {cls:<20s} {f1:.3f}{flag}")

    # Part 3
    print("\n⚖️  PART 3")
    result3 = improve_cancellation_recall(train_t, train_l, val_t, val_l)
    print(f"  Strategies: {result3['strategies_used']}")
    print(f"  Cancellation recall:    {result3['cancellation_recall']:.3f}")
    print(f"  Cancellation precision: {result3['cancellation_precision']:.3f}")
    print(f"  Macro F1:               {result3['val_macro_f1']:.3f}")
    print(f"  Tradeoff: {result3['tradeoff_analysis']}")

    # Part 4
    print("\n📋 PART 4")
    eval_result = full_evaluation(result3["model"], test_t, test_l, label_names)
    print(f"  Test accuracy:    {eval_result['test_accuracy']}")
    print(f"  Test macro F1:    {eval_result['test_macro_f1']}")
    print(f"  Test weighted F1: {eval_result['test_weighted_f1']}")
    print(f"\n  Confusion matrix (rows=true, cols=pred):")
    print(f"  Labels: {label_names}")
    for row in eval_result["confusion_matrix"]:
        print(f"    {row}")
    print(f"\n  Error analysis:")
    for err in eval_result["error_analysis"][:3]:
        print(f'    "{err["text"][:60]}..."')
        print(f'      {err["true"]} → {err["predicted"]}: {err["why"]}')

    design = production_design()
    print(f"\n  Production design answers:")
    for key, val in design.items():
        print(f"\n  [{key}]")
        print(f"  {val}")

    print("\n" + "=" * 70)


if __name__ == "__main__":
    main()